# MD-only BAT relabeling via OpenRouter (Kimi K2.5)

**Step 1** — filter `bat_posts_results_final_patched.csv` down to MD == "YES" rows, save `md_only.csv`.
**Step 2** — send each MD-only post through the BAT prompt on OpenRouter (`moonshotai/kimi-k2.5`), save `md_only_relabeled.csv`.

Run cells top to bottom. Step 2 is checkpointed to a `.jsonl` file, so if it dies partway you can just re-run that cell and it'll skip rows already completed.

## Config

In [1]:
import os
import json
import asyncio
from pathlib import Path

import pandas as pd
import httpx
from dotenv import load_dotenv
from tqdm.auto import tqdm

# ---- paths: adjust as needed ----
INPUT_BAT_CSV = "bat_posts_results_final_patched.csv"
MD_ONLY_CSV = "md_only.csv"
RELABELED_CSV = "md_only_relabeled.csv"
CHECKPOINT_PATH = Path("md_only_relabeled_checkpoint.jsonl")

# ---- OpenRouter config ----
MODEL = "moonshotai/kimi-k2.5"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MAX_TOKENS = 4000
MAX_CONCURRENCY = 20
MAX_RETRIES = 4

load_dotenv()  # picks up comment_data/.env if notebook is run from that directory
API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "OPENROUTER_API_KEY not found in environment / .env"
print("API key loaded OK")

API key loaded OK


/Users/nadia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — filter to MD == "YES" only

MD column naming has drifted a bit across files, so this checks a few candidates and prints the value counts before filtering.

In [2]:
def find_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find a {label} column. Columns present: {list(df.columns)}")

print(f"Reading {INPUT_BAT_CSV} ...")
bat_df = pd.read_csv(INPUT_BAT_CSV)
print(f"Loaded {len(bat_df):,} rows, {len(bat_df.columns)} columns.")

md_col = find_col(bat_df, ["MD", "md", "MD_flag", "MD_label"], "MD")
print(f"Using MD column: '{md_col}'")

normalized = bat_df[md_col].astype(str).str.strip().str.upper()
print("\nValue counts for MD column:")
print(normalized.value_counts(dropna=False).to_string())

md_df = bat_df[normalized == "YES"].copy()
n_md = len(md_df)
pct = 100 * n_md / len(bat_df) if len(bat_df) else 0
print(f"\nMD == YES: {n_md:,} rows out of {len(bat_df):,} total ({pct:.2f}%)")

md_df.to_csv(MD_ONLY_CSV, index=False)
print(f"Saved MD-only subset to {MD_ONLY_CSV}")
md_df.head()

Reading bat_posts_results_final_patched.csv ...
Loaded 144,652 rows, 16 columns.
Using MD column: 'MD'

Value counts for MD column:
MD
NO     141258
YES      3394

MD == YES: 3,394 rows out of 144,652 total (2.35%)
Saved MD-only subset to md_only.csv


,row_type,post_id,comment_id,text,triage,na_subtype,triage_reason,EX,EMO,COG,MD,bat_score,EX_reasoning,EMO_reasoning,COG_reasoning,MD_reasoning
37,post,12t87xb,NaN,Am I the Only One...\nAm I the only one who ge...,NaN,NaN,NaN,NO,NO,NO,YES,1,"No mention of physical tiredness, mental deple...","No intense emotional reactions, irritability, ...","No indication of brain fog, memory issues, dif...","""jaded me"" and ""Is that all, really?"" indicati..."
90,post,1k7d8py,NaN,Burnout - How to leave cyber security entirely...,NaN,NaN,NaN,YES,YES,NO,YES,3,"""So burned out"", ""So tired of the constant bat...","""tired of the constant battles"", ""atrocious"", ...",While the author expresses career indecision a...,"""I am just... done"", ""thinking of leaving info..."
111,post,1nfa5d7,NaN,Retirement\nSo i am retiring from the public s...,NaN,NaN,NaN,YES,YES,NO,YES,3,35 years of grinding tech,throwing my laptop into a volcano,No work-related cognitive symptoms such as bra...,"throwing my laptop into a volcano, and not tou..."
123,post,1oivdvr,NaN,Legal and Compliance challenges... Time to run...,NaN,NaN,NaN,NO,YES,NO,YES,2,"No mention of energy depletion, fatigue, or su...","""no way in hell"", ""squirrely"", and ""complete b...","No difficulty with memory, focus, or decision-...","""Time to run away"", ""really considering walkin..."
198,post,7rmj6f,NaN,Where do I stand for jobs and what should I do...,NaN,NaN,NaN,NO,NO,NO,YES,1,"No mention of energy depletion, physical tired...",While the user expresses that the job search i...,"No mention of memory problems, brain fog, trou...",not wanting to do anything


## Step 2 — BAT prompt

In [3]:
BAT_INSTRUCTIONS = """You are a researcher applying the Burnout Assessment Tool (BAT) to Reddit text.
Your job is to decide YES or NO for each of four burnout dimensions.  We will consider any form of burnout
and stress signal written in any tense such as past, present and future. For example: "I will feel stress..",
"I have gone through a lot of stress or I was stressed or burnout", "i am having sleep trouble or having
trouble balancing my work and personal life".


IMPORTANT RULES BEFORE YOU START:
- Read the text cold, with no assumptions about whether burnout is present.
- This is a SENSITIVITY-FIRST task. When in doubt, lean YES.
 Missing a true burnout signal is worse than flagging a stress-adjacent one.
- A post asking others about their experience is NOT the same as expressing it yourself.
- Naturalistic Reddit language rarely uses clinical terms — look for the meaning, not exact words.
- The word "burnout" alone with no other signal = NO. Any supporting signal alongside it = YES.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EX — Exhaustion
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Definition: Energy loss from work — physical (tiredness, feeling weak) AND/OR mental
(feeling drained, worn-out). Includes sustained overload that implies depletion even
without the exact word "drained".


YES if any of:
 - Explicit depletion: "drained", "nothing left", "used up", "running on empty",
   "physically broken", "can't decompress", "sleep doesn't help"
 - Sustained overload personally described over weeks or months:
   "tough couple of years", "impossible deadlines", "never ending [workload]",
   "always something left to do", chronic on-call or work pressure as personal cost
 - Physical or health deterioration attributed to work:
   "health has been deteriorating [since this job]", "getting sick from work"
 - Persistent misery tied to the job: "been miserable since [starting this role]"
 - Lack of energy to start work, feeling completely used up after working


NO if:
 - A single mentioned bad day with no sustained element
 - Asking others if they experience exhaustion (not expressing it personally)
 - Only boredom or dissatisfaction with no energy or health cost described


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EMO — Emotional Impairment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Definition: Intense, persistent, or disproportionate emotional reactions tied to work.
Does NOT require explicit "loss of control" — strong sustained negative emotion qualifies.


YES if any of:
 - Strong hate or intense aversion toward the work situation:
   "I HATE [job/role/situation]", "this job is making me miserable"
 - Repeated or stacked emotional signals in the same post indicating sustained distress
   (e.g., "punch in the face" used twice, or multiple frustration phrases together)
 - Snapping, crying unexpectedly, or overreacting at work
 - Feeling upset, sad, or angry without a clear single cause
 - Persistent irritability or frustration tied to work beyond one incident
 - Feeling frustrated and angry at work, feeling upset or sad without knowing why
 - Feeling unable to control one's emotions at work, irritability, overreacting


NO if:
 - A single proportionate frustration about one event mentioned once and calmly
 - Mild annoyance described without intensity or repetition
 - Asking others if they feel frustrated (not expressing it personally)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
COG — Cognitive Impairment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Definition: Difficulty with memory, focus, or decision-making at work.
Includes feeling cognitively overwhelmed by job demands (volume, complexity, pace).


YES if any of:
 - Feeling overwhelmed by cognitive demands: volume of alerts, tasks, or complexity
   (even in a newer role, if the overwhelm goes beyond normal new-job adjustment)
 - Brain fog, forgetting procedures or tasks, trouble concentrating
 - Indecision or inability to make decisions that would normally be easy
 - Difficulty learning or keeping up with what the job demands
 - Being absent-minded, forgetful,for mentally scattered at work
 - Difficulties thinking clearly, poor memory, attention or concentration at work


NO if:
 - Brand new to role AND describes normal learning difficulty with no signs of distress
 - Asking others about cognitive difficulty without expressing it personally
 - Feeling generally confused with no work-specific cognitive symptom


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MD — Mental Distance
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Definition: Persistent psychological withdrawal — indifference, cynicism, aversion, autopilot.


YES if any of:
 - Explicit loss of meaning or interest: "what's the point", "don't care anymore",
   "I used to love this but now feel nothing", "no longer want to"
 - Going through the motions or autopilot described personally
 - Active avoidance of work tasks or colleagues
 - Persistent cynical or resentful tone throughout the post
 - Persistent dread of work: "I dread going in / this role / these tasks"
 - Withdrawing mentally or physically from work, avoiding contact with colleagues


NO if:
 - Asking others about engagement (not expressing own detachment)
 - Single bad day or one-off complaint
 - Considering career change out of ambition or curiosity (active agency, not withdrawal)
 - Mild boredom mentioned once without sustained withdrawal signals


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Respond with JSON only. No markdown fences. No explanation outside the JSON.


For each category provide a reasoning field explaining the decision whether YES or NO:
 - If YES: quote the exact phrase from the text that triggered YES.
 - If NO:  write one sentence explaining why the text did not meet the threshold.


{
 "EX":          "YES" or "NO",
 "EMO":         "YES" or "NO",
 "COG":         "YES" or "NO",
 "MD":          "YES" or "NO",
 "EX_reasoning":  "quoted phrase if YES  /  one-sentence explanation if NO",
 "EMO_reasoning": "quoted phrase if YES  /  one-sentence explanation if NO",
 "COG_reasoning": "quoted phrase if YES  /  one-sentence explanation if NO",
 "MD_reasoning":  "quoted phrase if YES  /  one-sentence explanation if NO"
}"""
print(f"Prompt loaded, {len(BAT_INSTRUCTIONS):,} chars")

Prompt loaded, 6,213 chars


## Step 2 — load `md_only.csv` and detect text/id columns

Check the printed column name below — the auto-detection guesses from common candidates, but confirm it picked the right one for this file before running the full batch.

In [4]:
TEXT_COL_CANDIDATES = ["cleaned_body", "body", "text", "selftext", "post_text"]
ID_COL_CANDIDATES = ["id", "post_id"]

label_df = pd.read_csv(MD_ONLY_CSV)
print(f"Loaded {len(label_df):,} rows from {MD_ONLY_CSV}")

id_col = find_col(label_df, ID_COL_CANDIDATES, "id")
text_col = find_col(label_df, TEXT_COL_CANDIDATES, "text")
print(f"Using id column: '{id_col}', text column: '{text_col}'")

def build_text(row):
    parts = []
    if "title" in row.index and pd.notna(row["title"]):
        parts.append(str(row["title"]))
    body_val = row.get(text_col)
    if pd.notna(body_val):
        parts.append(str(body_val))
    return "\n\n".join(p for p in parts if p.strip())

label_df[["title"]].head() if "title" in label_df.columns else label_df[[text_col]].head()

Loaded 3,394 rows from md_only.csv
Using id column: 'post_id', text column: 'text'


,text
0,Am I the Only One...\nAm I the only one who ge...
1,Burnout - How to leave cyber security entirely...
2,Retirement\nSo i am retiring from the public s...
3,Legal and Compliance challenges... Time to run...
4,Where do I stand for jobs and what should I do...


## Step 2 — quick single-row test

Sanity-check one call before committing the full budget.

In [5]:
async def call_openrouter(client, text):
    payload = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": BAT_INSTRUCTIONS},
            {"role": "user", "content": text},
        ],
    }
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = await client.post(OPENROUTER_URL, json=payload, headers=headers, timeout=120)
            resp.raise_for_status()
            data = resp.json()
            content = data["choices"][0]["message"]["content"].strip()
            if content.startswith("```"):
                content = content.strip("`")
                if content.lower().startswith("json"):
                    content = content[4:].strip()
            return json.loads(content)
        except Exception as e:
            last_err = e
            await asyncio.sleep(min(2 ** attempt, 20))
    return {"error": str(last_err)}

test_row = label_df.iloc[0]
test_text = build_text(test_row)
print(test_text[:500], "...\n")

async with httpx.AsyncClient() as client:
    test_result = await call_openrouter(client, test_text)

test_result

Am I the Only One...
Am I the only one who gets a pen test report sometimes, and asks themselves "Is that all, really?"

Maybe spending 7+ years as a pen tested has jaded me, but as a CISO I look at these reports and just have to wonder. Are we finally getting that good at writing apps, or are we that bad at pen testing? ...



{'EX': 'NO',
 'EMO': 'NO',
 'COG': 'NO',
 'MD': 'NO',
 'EX_reasoning': 'The post expresses professional skepticism but describes no physical or mental energy depletion, sustained overload, or health deterioration.',
 'EMO_reasoning': "The text contains mild, measured skepticism ('Is that all, really?') and a single mention of being 'jaded' without intense, persistent, or disproportionate emotional reactions.",
 'COG_reasoning': 'There is no indication of brain fog, forgetfulness, trouble concentrating, or feeling cognitively overwhelmed by job demands.',
 'MD_reasoning': "While the author mentions being 'jaded,' this is a single, mild expression of cynicism without explicit withdrawal, autopilot, loss of meaning, or persistent disengagement from the work."}

## Step 2 — run the full batch

Concurrency capped at `MAX_CONCURRENCY`, results streamed to `CHECKPOINT_PATH` as they complete. Safe to interrupt and re-run — already-completed ids are skipped.

In [6]:
async def process_row(sem, client, row_id, text, results, pbar):
    async with sem:
        result = await call_openrouter(client, text)
    result["_id"] = row_id
    results.append(result)
    pbar.update(1)

async def run_batch(df):
    done_ids = set()
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r") as f:
            for line in f:
                if line.strip():
                    done_ids.add(json.loads(line)["_id"])
        print(f"Resuming: {len(done_ids):,} rows already completed in {CHECKPOINT_PATH}")

    todo = df[~df[id_col].astype(str).isin(done_ids)]
    print(f"{len(todo):,} rows to label ({len(done_ids):,} already done, {len(df):,} total).")
    if len(todo) == 0:
        return

    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    results = []
    async with httpx.AsyncClient() as client:
        with tqdm(total=len(todo)) as pbar:
            tasks = [
                process_row(sem, client, str(row[id_col]), build_text(row), results, pbar)
                for _, row in todo.iterrows()
            ]
            batch_size = 200
            for i in range(0, len(tasks), batch_size):
                batch = tasks[i : i + batch_size]
                await asyncio.gather(*batch)
                with open(CHECKPOINT_PATH, "a") as f:
                    for r in results[-len(batch):]:
                        f.write(json.dumps(r) + "\n")

await run_batch(label_df)

3,394 rows to label (0 already done, 3,394 total).


100%|██████████| 3394/3394 [6:04:07<00:00,  6.44s/it]      


## Step 2 — merge checkpoint results back onto `md_only.csv` and save

In [7]:
records = []
with open(CHECKPOINT_PATH, "r") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

results_df = pd.DataFrame(records).rename(columns={"_id": id_col})
results_df = results_df.add_suffix("_llm").rename(columns={f"{id_col}_llm": id_col})
results_df[id_col] = results_df[id_col].astype(str)

merged = label_df.copy()
merged[id_col] = merged[id_col].astype(str)
merged = merged.merge(results_df, on=id_col, how="left")

merged.to_csv(RELABELED_CSV, index=False)
print(f"Saved {len(merged):,} labeled rows to {RELABELED_CSV}")
merged.head()

Saved 3,394 labeled rows to md_only_relabeled.csv


,row_type,post_id,comment_id,text,triage,na_subtype,triage_reason,EX,EMO,COG,...,MD_reasoning,EX_llm,EMO_llm,COG_llm,MD_llm,EX_reasoning_llm,EMO_reasoning_llm,COG_reasoning_llm,MD_reasoning_llm,error_llm
0,post,12t87xb,NaN,Am I the Only One...\nAm I the only one who ge...,NaN,NaN,NaN,NO,NO,NO,...,"""jaded me"" and ""Is that all, really?"" indicati...",NO,NO,NO,YES,Text contains no mention of physical tiredness...,"No intense emotional reactions, irritability, ...","No difficulties with memory, focus, decision-m...","""jaded"" and questioning ""Is that all, really?""...",NaN
1,post,1k7d8py,NaN,Burnout - How to leave cyber security entirely...,NaN,NaN,NaN,YES,YES,NO,...,"""I am just... done"", ""thinking of leaving info...",YES,YES,NO,YES,So burned out. So tired of the constant battle...,when you add in the lack of staff and the need...,The text does not describe work-related cognit...,I've come to the realization that I am just......,NaN
2,post,1nfa5d7,NaN,Retirement\nSo i am retiring from the public s...,NaN,NaN,NaN,YES,YES,NO,...,"throwing my laptop into a volcano, and not tou...",YES,NO,NO,YES,after 35 years of grinding tech,The post expresses retirement ambivalence and ...,No work-related cognitive symptoms such as bra...,"throwing my laptop into a volcano, and not tou...",NaN
3,post,1oivdvr,NaN,Legal and Compliance challenges... Time to run...,NaN,NaN,NaN,NO,YES,NO,...,"""Time to run away"", ""really considering walkin...",NO,NO,NO,NO,"No signals of physical depletion, mental exhau...","While the situation involves conflict, the pos...","No indications of brain fog, forgetfulness, co...",Considering resignation due to ethical concern...,NaN
4,post,7rmj6f,NaN,Where do I stand for jobs and what should I do...,NaN,NaN,NaN,NO,NO,NO,...,not wanting to do anything,NO,NO,NO,NO,The text describes unemployment and job-search...,The poster fears future depression but does no...,The poster asks for career advice but does not...,'Not wanting to do anything' is framed as a fu...,NaN


## Optional — quick agreement check

Compares the original BAT MD/EX/EMO/COG labels against the new `*_llm` labels, since these should mostly agree (all input rows were already MD == YES).

In [8]:
for construct in ["EX", "EMO", "COG", "MD"]:
    orig_col = construct if construct in merged.columns else None
    llm_col = f"{construct}_llm"
    if orig_col and llm_col in merged.columns:
        orig = merged[orig_col].astype(str).str.strip().str.upper()
        llm = merged[llm_col].astype(str).str.strip().str.upper()
        agree = (orig == llm).mean()
        print(f"{construct}: {agree:.1%} agreement between original BAT label and Kimi relabel")

EX: 87.4% agreement between original BAT label and Kimi relabel
EMO: 85.6% agreement between original BAT label and Kimi relabel
COG: 91.7% agreement between original BAT label and Kimi relabel
MD: 70.6% agreement between original BAT label and Kimi relabel


In [10]:
failed_ids = []
import json
from pathlib import Path
CHECKPOINT_PATH = Path("md_only_relabeled_checkpoint.jsonl")
with open(CHECKPOINT_PATH, "r") as f:
    lines = f.readlines()
for line in lines:
    rec = json.loads(line)
    if "error" in rec:
        failed_ids.append(rec["_id"])
print(failed_ids)

kept = [l for l in lines if json.loads(l)["_id"] not in failed_ids]
with open(CHECKPOINT_PATH, "w") as f:
    f.writelines(kept)

await run_batch(label_df)

[]
Resuming: 3,386 rows already completed in md_only_relabeled_checkpoint.jsonl
8 rows to label (3,386 already done, 3,394 total).


100%|██████████| 8/8 [03:42<00:00, 27.78s/it]


In [11]:
records = []
with open(CHECKPOINT_PATH, "r") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

results_df = pd.DataFrame(records).rename(columns={"_id": id_col})
results_df = results_df.add_suffix("_llm").rename(columns={f"{id_col}_llm": id_col})
results_df[id_col] = results_df[id_col].astype(str)

merged = label_df.copy()
merged[id_col] = merged[id_col].astype(str)
merged = merged.merge(results_df, on=id_col, how="left")

merged.to_csv(RELABELED_CSV, index=False)
print(f"Saved {len(merged):,} labeled rows to {RELABELED_CSV}")

Saved 3,394 labeled rows to md_only_relabeled.csv


In [12]:
md_llm_norm = merged["MD_llm"].astype(str).str.strip().str.upper()
print(md_llm_norm.value_counts())

MD_llm
YES    2399
NO      995
Name: count, dtype: int64
